# STEP 3: Fetch Supreme Court Judgment Metadata (Public S3)

Source: `s3://indian-supreme-court-judgments` (public, region `us-east-1`).

This notebook fetches Supreme Court metadata parquet by year, applies ADR keyword signals, and saves a final parquet for downstream labeling.

## Notes
- Designed for Google Colab.
- Uses per-year part files + streaming combine to stay RAM-safe.
- Includes robust mixed-date parsing to avoid `dayfirst` warnings.

In [18]:
# Colab setup
%pip install -q pandas pyarrow

import pandas as pd
from pathlib import Path
import time
import gc
import pyarrow as pa
import pyarrow.parquet as pq
from collections import Counter

In [19]:
# Connect Google Drive (Colab)
from google.colab import drive

drive.mount('/content/drive')
print('Connected to the drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Connected to the drive


In [20]:
# Configuration
BASE_DIR = Path('/content/drive/MyDrive/MiniProject')
OUTPUT_DIR = BASE_DIR / 'compiled_dataset'
SHARD_DIR = OUTPUT_DIR / 'sc_metadata_parts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)

SC_S3_BASE = 'https://indian-supreme-court-judgments.s3.amazonaws.com'
TARGET_YEARS = list(range(2000, 2025))  # 2000-2024
REQUEST_DELAY_SECONDS = 0.2

ADR_KEYWORDS = [
    'arbitration', 'mediation', 'conciliation', 'lok adalat',
    'settlement', 'negotiation', 'adr', 'odr',
    'section 89', 'arbitration and conciliation',
]

In [21]:
def fetch_sc_parquet(year: int, verbose: bool = False) -> pd.DataFrame:
    url = f"{SC_S3_BASE}/metadata/parquet/year={year}/metadata.parquet"
    try:
        df = pd.read_parquet(url)
        df['source_year'] = year
        df['court_level'] = 'Supreme Court'
        return df
    except Exception as e:
        if verbose:
            print(f"error: {type(e).__name__}: {e}")
        return pd.DataFrame()


def _parse_mixed_date(series: pd.Series) -> pd.Series:
    """Parse mixed date strings while handling dd-mm-yyyy safely."""
    s = series.astype('string').str.strip()

    parsed = pd.to_datetime(s, errors='coerce', dayfirst=True)

    missing = parsed.isna() & s.notna() & (s != '')
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(
            s.loc[missing], errors='coerce', format='%Y-%m-%d'
        )

    return parsed


def prepare_columns(df: pd.DataFrame) -> pd.DataFrame:
    for datecol in ['decision_date', 'date_of_registration']:
        if datecol in df.columns:
            df[datecol] = _parse_mixed_date(df[datecol])

    if 'title' in df.columns:
        title_lower = df['title'].astype('string').str.lower().fillna('')
        df['adr_keyword_in_title'] = title_lower.apply(
            lambda t: any(kw in t for kw in ADR_KEYWORDS)
        )

    if 'description' in df.columns:
        desc_lower = df['description'].astype('string').str.lower().fillna('')
        df['adr_keyword_in_desc'] = desc_lower.apply(
            lambda d: any(kw in d for kw in ADR_KEYWORDS)
        )

    return df


def fetch_and_save_parts():
    print('=' * 60)
    print('Supreme Court S3 Metadata Fetcher (Part Writer)')
    print('=' * 60)
    print(f"Fetching years: {TARGET_YEARS[0]}-{TARGET_YEARS[-1]}")
    print(f"Total requests to attempt: {len(TARGET_YEARS)}\n")

    written_parts = []
    total_rows = 0

    for year in TARGET_YEARS:
        print(f"  Year {year} ...", end=' ', flush=True)
        df = fetch_sc_parquet(year, verbose=True)

        if df.empty:
            print('no data')
        else:
            df = prepare_columns(df)
            part_path = SHARD_DIR / f'sc_{year}.parquet'
            df.to_parquet(part_path, index=False)
            written_parts.append(part_path)
            total_rows += len(df)
            print(f"{len(df):,} rows -> {part_path.name}")

            del df
            gc.collect()

        time.sleep(REQUEST_DELAY_SECONDS)

    print(f"\nPart files written: {len(written_parts)}")
    print(f"Total rows kept: {total_rows:,}")
    return written_parts


def _build_canonical_schema(parts):
    """Create one schema that safely handles null/string drift across yearly parts."""
    all_cols = set()
    for p in parts:
        all_cols.update(pq.read_schema(p).names)

    ordered = sorted(all_cols)

    bool_cols = {'adr_keyword_in_title', 'adr_keyword_in_desc'}
    int_cols = {'source_year'}
    ts_cols = {'decision_date', 'date_of_registration'}

    fields = []
    for col in ordered:
        if col in bool_cols:
            fields.append(pa.field(col, pa.bool_()))
        elif col in int_cols:
            fields.append(pa.field(col, pa.int64()))
        elif col in ts_cols:
            fields.append(pa.field(col, pa.timestamp('ns')))
        else:
            fields.append(pa.field(col, pa.string()))

    return pa.schema(fields), ordered


def _normalize_part_df(df, ordered_cols):
    """Normalize dataframe dtypes to match canonical schema before writing."""
    for col in ordered_cols:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[ordered_cols]

    for col in ordered_cols:
        if col in ['decision_date', 'date_of_registration']:
            df[col] = pd.to_datetime(df[col], errors='coerce')
        elif col in ['source_year']:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
        elif col in ['adr_keyword_in_title', 'adr_keyword_in_desc']:
            df[col] = df[col].astype('boolean')
        else:
            df[col] = df[col].astype('string')

    return df


def combine_parts_to_final(parts):
    """Combine part files using streaming parquet writes (low RAM + schema-safe)."""
    if not parts:
        print('\n[ERROR] No SC part files to combine.')
        return None

    print(f"\n[Combining] {len(parts)} part files (streaming mode) ...")
    out_path = OUTPUT_DIR / 'sc_metadata.parquet'

    schema, ordered_cols = _build_canonical_schema(parts)

    writer = None
    total_rows = 0
    decade_counter = Counter()
    title_adr_true = 0

    for i, p in enumerate(sorted(parts), start=1):
        df_small = pd.read_parquet(p)
        df_small = _normalize_part_df(df_small, ordered_cols)

        table = pa.Table.from_pandas(df_small, schema=schema, preserve_index=False, safe=False)
        table = table.replace_schema_metadata(None)

        if writer is None:
            # Use first normalized table schema to avoid metadata mismatch.
            writer = pq.ParquetWriter(out_path, table.schema)

        writer.write_table(table)
        total_rows += table.num_rows

        if 'source_year' in df_small.columns:
            decades = (df_small['source_year'] // 10 * 10).astype('Int64').astype('string') + 's'
            decade_counter.update(decades.dropna().tolist())
        if 'adr_keyword_in_title' in df_small.columns:
            title_adr_true += int(df_small['adr_keyword_in_title'].fillna(False).sum())

        del df_small
        del table
        gc.collect()

        if i % 10 == 0 or i == len(parts):
            print(f"  merged {i}/{len(parts)} parts ...")

    if writer is not None:
        writer.close()

    print(f"Saved -> {out_path}")
    print(f"Rows: {total_rows:,}")
    print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")

    print('\n-- Summary --')
    print(f"Cases with ADR keyword in title: {title_adr_true:,}")

    if decade_counter:
        print('\nRows per decade:')
        print(pd.Series(decade_counter).sort_index())

    gc.collect()
    return out_path

In [22]:
# Run Step 3
# Reuse existing part files if present; fetch only if missing.
parts = sorted(SHARD_DIR.glob('sc_*.parquet'))
if not parts:
    parts = fetch_and_save_parts()

final_path = combine_parts_to_final(parts)
print('\nDone')


[Combining] 25 part files (streaming mode) ...
  merged 10/25 parts ...
  merged 20/25 parts ...
  merged 25/25 parts ...
Saved -> /content/drive/MyDrive/MiniProject/compiled_dataset/sc_metadata.parquet
Rows: 20,992
File size: 23.0 MB

-- Summary --
Cases with ADR keyword in title: 129

Rows per decade:
2000s    9104
2010s    7954
2020s    3934
dtype: int64

Done


In [23]:
# Copy Supreme Court dataset to dedicated Drive folder
import shutil

TARGET_DIR = Path('/content/drive/MyDrive/MiniProject/ISCJ_dataset_180426')
TARGET_DIR.mkdir(parents=True, exist_ok=True)

combined_file = OUTPUT_DIR / 'sc_metadata.parquet'
if combined_file.exists():
    shutil.copy2(combined_file, TARGET_DIR / combined_file.name)
    print(f'Copied: {combined_file.name} -> {TARGET_DIR}')
else:
    print('Combined file not found; skipping sc_metadata.parquet copy')

parts_target = TARGET_DIR / 'sc_metadata_parts'
if SHARD_DIR.exists():
    shutil.copytree(SHARD_DIR, parts_target, dirs_exist_ok=True)
    print(f'Copied parts folder -> {parts_target}')
else:
    print('Parts folder not found; nothing to copy')

print('Upload to Drive folder ISCJ_dataset_180426 completed')

Copied: sc_metadata.parquet -> /content/drive/MyDrive/MiniProject/ISCJ_dataset_180426
Copied parts folder -> /content/drive/MyDrive/MiniProject/ISCJ_dataset_180426/sc_metadata_parts
Upload to Drive folder ISCJ_dataset_180426 completed
